# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset DOI:** [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)

**License:** [Open Data Commons Attribution](https://opendatacommons.org/licenses/by/1-0/)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset name: {metadata.get('name')}")
print(f"Description: {metadata.get('description')}")
print(f"Version: {metadata.get('version')}")
print(f"License: {metadata.get('license')}")
print(f"Fields: {list(metadata.keys())}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema describes tables as "record sets". Each record set typically corresponds to a main table in a data package. Let's display the available record sets, and for each record set, list its fields (`@id`s).

In [ ]:
def _pretty_print_recordsets(ds):
    record_sets = ds.record_sets
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"  Record set name: {rs.name}")
        print(f"    @id: {rs.id}")
        print(f"    Description: {getattr(rs, 'description', None)}")
        if hasattr(rs, 'fields'):
            print("    Fields (with @id):")
            for f in rs.fields:
                print(f"      - {getattr(f, 'name', '')} (@id: {getattr(f, 'id', '')})  type: {getattr(f, 'data_type', '')}")
        print()
    return record_sets

record_sets = _pretty_print_recordsets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above to specify which tables you wish to analyze. Here, we extract all available record sets and place each in a DataFrame. For demonstration, we show the columns and the first few records of the main protein record set.

In [ ]:
# Build dictionary of DataFrames for each record set using @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    # Load records via the unique @id
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df

print("\nAvailable record set @ids:")
print(record_set_ids)

# Print columns of main record set (if exists)
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nColumns in record set {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    print("\nFirst 5 rows:")
    display(dataframes[example_rs_id].head())
else:
    print("No record sets found in the dataset!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates these operations for a selected numeric field, using `@id`s to ensure all references are schema-compliant.

In [ ]:
# For demonstration, pick the first record set and one of its numeric fields
# You can look up the field @id (see previous overview cell or Croissant schema)

# Select a record set to work with
if len(record_set_ids) > 0:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Display all columns to help select a numeric field
    print(f"Fields in record set {record_set_id}:")
    print(df.columns.tolist())

    # Try to identify some numeric field (e.g., 'MW' or 'Coverage (%)' by name or @id)
    # Here we heuristically pick the first numeric column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to convert a likely numeric column
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except:
                pass

    print(f"Using numeric field: {numeric_field}")

    # Set a filter threshold
    threshold = df[numeric_field].mean() if numeric_field else None
    if numeric_field and threshold is not None:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a column if present (e.g. 'description' or other categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_string_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}' (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found to analyze.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Below, we plot a histogram of the selected numeric field and a boxplot grouped by a categorical field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if len(record_set_ids) > 0 and numeric_field:
    # Histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if group_field exists
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=60)
        plt.show()
else:
    print("Not enough data for visualization, or no numeric field found.")

## 6. Conclusion
In this notebook, we:

- Loaded the FAIR² mass spectrometry dataset using the Croissant schema and `mlcroissant`.
- Explored the metadata, available record sets, and fields (referenced by their `@id`).
- Extracted tabular data for analysis.
- Applied simple EDA: filtering, normalization, grouping, and visualization of selected numeric fields.

You can now adapt this workflow to analyze other fields or record sets, or incorporate custom domain-specific analyses relevant to extracellular vesicle proteomics!